# Task07 — Colab LR Sweep 12 lượt (`phase2-v1` / `colab-task06-v1`)

Chọn **Runtime → Change runtime type → GPU** (ưu tiên A100 hoặc GPU hỗ trợ BF16), điền đường dẫn 2 checkpoint nguồn Stage 1 ở **Bước 1**, rồi chạy lần lượt từng cell theo thứ tự.

### Phạm vi thực nghiệm
- **Mục tiêu:** Dò learning rate (`run_mode="sweep"`, **chỉ dùng `train 10%` và `dev`**, `seed=0`, backbone `ViSoBERT`, `BF16 strict`) trên cùng một commit và cùng profile môi trường **`colab-task06-v1`**, sau đó gọi `src.phase2.select_lr` khóa `selection_p1_lr.json` và `p1_lr_selection.csv`.
- **12 cấu hình sweep:**
  - `DFT` (6 lượt): `SA` (`NEU-ESC 10%`) × `{2e-5, 3e-5, 5e-5}` và `SB` (`UIT-VSMEC 10%`) × `{2e-5, 3e-5, 5e-5}`.
  - `Seq` (6 lượt): `SA` (`VSMEC7 → NEU3`) × `{1e-5, 2e-5, 3e-5}` và `SB` (`NEU3 → VSMEC7`) × `{1e-5, 2e-5, 3e-5}`.
- **Cách ly tập `test`:** Không mở, không tiền xử lý và không sinh `predictions_test.csv` ở bất kỳ lượt nào.

### Lưu ý vận hành & phục hồi khi bị ngắt runtime
- **Nguồn checkpoint Stage 1 (`Seq`):** Cần đúng 2 checkpoint `ViSoBERT` `seed=0` (`UIT-VSMEC` 7 lớp và `NEU-ESC` 3 lớp) trên Google Drive; tuyệt đối không dùng checkpoint smoke Task06 làm nguồn `Seq`.
- **Phục hồi xuyên phiên Colab:** Toàn bộ quá trình train diễn ra trên SSD `/content` để tránh nghẽn I/O `resume.pt` (~1,1 GB), đồng thời đồng bộ gia tăng kết quả mỗi lượt hoàn tất lên Google Drive (`USE_DRIVE_BACKUP=True`). Nếu bị thu hồi runtime giữa chừng:
  - Giữ nguyên `PROFILE_ID`.
  - Nếu nhánh `phase2-resource-transfer` trên GitHub đã có thêm commit mới, đặt `REPO_REF` bằng **SHA 40 ký tự** ghi trong `environment.json` của lượt trước để giữ nguyên `fingerprint`, tự khôi phục từ Drive và `SKIP` ngay các lượt đã xong.
- **Bảo mật token:** Repo riêng tư bật `PRIVATE_REPO=True` và thêm Colab Secret `GITHUB_TOKEN`. Có thể thêm Colab Secret `HF_TOKEN` (tùy chọn) để tránh giới hạn tải ẩn danh `uitnlp/visobert`. Tuyệt đối không dán token vào URL hay code.


## 1. Tham số, profile môi trường và đường dẫn cố định
Khai báo tập trung toàn bộ cấu hình phiên chạy. `RESULTS_DIR`, `CKPT_OUT_DIR` và `SOURCE_CKPT_STAGE_DIR` dùng đường dẫn cố định theo `<commit[:12]>/<profile_id>` (không chèn `SESSION_ID` vào đường dẫn thư mục để không làm lệch `fingerprint` khi nối phiên; `SESSION_ID` chỉ gắn vào tên file log/môi trường).


In [15]:
from pathlib import Path
import csv, hashlib, json, os, platform, re, shutil, signal, subprocess, sys, time, zipfile

T0_SETUP = time.perf_counter()
SESSION_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())

REPO_URL = "https://github.com/PhuongNgoVan/Xdomain-Transfer-Learning.git"
REPO_REF = "phase2-resource-transfer"  # đổi sang commit SHA 40 ký tự nếu nối lại phiên cũ sau khi nhánh có commit mới
PRIVATE_REPO = False

PROFILE_ID = "colab-task06-v1"
PROFILES = {
    "colab-task06-v1": {
        "python": "3.13.18",
        "torch": "2.14.1+cu130",
        "cuda": "13.0",
        "pins": {
            "transformers": "4.57.1",
            "huggingface-hub": "0.36.2",
            "tokenizers": "0.22.2",
            "numpy": "2.1.3",
            "pandas": "3.1.0",
            "scikit-learn": "1.6.1",
            "scipy": "1.16.3",
            "PyYAML": "6.0.2",
            "sentencepiece": "0.2.2",
            "protobuf": "7.36.2",
            "safetensors": "0.8.0",
            "tqdm": "4.67.3",
        },
    },
}
assert PROFILE_ID in PROFILES, f"Unknown PROFILE_ID: {PROFILE_ID}"

MOUNT_DRIVE = True
# Chọn 1 trong 2 cách chỉ định nguồn checkpoint Stage 1 (ViSoBERT, seed 0):
# Cách 1: 1 thư mục gốc chứa cả hai miền vsmec và neu3
DRIVE_SOURCE_CKPT_ROOT = ""      # ví dụ: "/content/drive/MyDrive/Xdomain/Stage1_Checkpoints"
# Cách 2: 2 thư mục riêng (dùng khi DRIVE_SOURCE_CKPT_ROOT để trống)
DRIVE_VSMEC_CKPT_DIR = "/content/drive/MyDrive/Xdomain/finetune-vsmec-7c/visobert/ckpt_baseline"
DRIVE_NEUESC_CKPT_DIR = "/content/drive/MyDrive/Xdomain/finetune-neu-3c/visobert/ckpt_baseline"

USE_DRIVE_BACKUP = True
DRIVE_BACKUP_BASE = Path("/content/drive/MyDrive/Xdomain/Phase2/Task07")
CLEAN_COMPLETED_RESUME_PT = True  # xóa resume.pt (~1.1 GB) sau khi lượt đã complete=True; giữ nguyên provenance.json

assert Path("/content").is_dir(), "Notebook này yêu cầu môi trường Google Colab (/content)."
REPO_DIR = Path("/content/xdomain-task07")
VENV_DIR = Path(f"/content/phase2-task07-{PROFILE_ID}-venv")
SOURCE_CKPT_STAGE_DIR = Path("/content/phase2_source_ckpts")
assert "sweep" not in str(SOURCE_CKPT_STAGE_DIR).lower() and "smoke" not in str(SOURCE_CKPT_STAGE_DIR).lower(), (
    "SOURCE_CKPT_STAGE_DIR không được chứa 'sweep' hoặc 'smoke' (src/zs/ckpt.py sẽ loại bỏ)."
)
OUTPUT_BASE = Path("/content/task07_outputs")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
print(f"Session: {SESSION_ID} | Profile: {PROFILE_ID}")


Session: 20261127T113132Z | Profile: colab-task06-v1


## 2. Kiểm tra GPU CUDA/BF16 sơ bộ ở kernel và mount Google Drive
Kiểm tra nhanh phần cứng, phiên bản Torch/CUDA nền của Colab (không cấp phát tensor `.cuda()` ở kernel để không chiếm giữ VRAM trong suốt 12 lượt chạy subprocess) và xác nhận đường dẫn nguồn checkpoint trên Drive trước khi tốn thời gian clone/cài gói.


In [16]:
import torch

assert torch.cuda.is_available(), "Chưa bật GPU CUDA trên Colab (Runtime -> Change runtime type -> GPU)."
assert torch.cuda.is_bf16_supported(), "GPU hiện tại không hỗ trợ BF16 strict; hãy chọn GPU khác (ví dụ A100/L4)."

gpu_name = torch.cuda.get_device_name(0)
vram_gib = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
print(f"GPU: {gpu_name} | VRAM (GiB): {vram_gib}")
print(
    f"Kernel Python: {platform.python_version()} | "
    f"Preinstalled Torch: {torch.__version__} | CUDA: {torch.version.cuda} | BF16: True"
)

expected_profile = PROFILES[PROFILE_ID]
assert torch.__version__ == expected_profile["torch"], (
    f"Torch trên Colab ({torch.__version__}) khác profile {PROFILE_ID} ({expected_profile['torch']}). "
    "Dừng trước khi cài đặt: ghi nhận khác biệt và tạo profile mới thay vì tự ép cài lại CUDA/Torch."
)
assert torch.version.cuda == expected_profile["cuda"], (
    f"CUDA của Torch ({torch.version.cuda}) khác profile {PROFILE_ID} ({expected_profile['cuda']})."
)
subprocess.run(["nvidia-smi"], check=True)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if DRIVE_SOURCE_CKPT_ROOT.strip():
    root_p = Path(DRIVE_SOURCE_CKPT_ROOT.strip())
    CKPT_SEARCH_ROOTS = {"vsmec": root_p, "neu3": root_p}
elif DRIVE_VSMEC_CKPT_DIR.strip() and DRIVE_NEUESC_CKPT_DIR.strip():
    CKPT_SEARCH_ROOTS = {
        "vsmec": Path(DRIVE_VSMEC_CKPT_DIR.strip()),
        "neu3": Path(DRIVE_NEUESC_CKPT_DIR.strip()),
    }
else:
    raise RuntimeError(
        "Chưa điền đường dẫn checkpoint nguồn Stage 1 (seed 0) ở Bước 1. "
        "Hãy điền DRIVE_SOURCE_CKPT_ROOT hoặc cả hai biến DRIVE_VSMEC_CKPT_DIR và DRIVE_NEUESC_CKPT_DIR."
    )

for dom, folder in CKPT_SEARCH_ROOTS.items():
    if not folder.is_dir():
        raise FileNotFoundError(f"Không tìm thấy thư mục checkpoint nguồn ({dom}): {folder}")
    print(f"Verified source search root ({dom}): {folder}")


GPU: NVIDIA L4 | VRAM (GiB): 22.03
Kernel Python: 3.13.16 | Preinstalled Torch: 2.14.1+cu130 | CUDA: 13.0 | BF16: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verified source search root (vsmec): /content/drive/MyDrive/Xdomain/finetune-vsmec-7c/visobert/ckpt_baseline
Verified source search root (neu3): /content/drive/MyDrive/Xdomain/finetune-neu-3c/visobert/ckpt_baseline


## 3. Fetch repository, cố định commit, chụp `frozen_before.json` và khôi phục tiến độ từ Drive
Fetch `REPO_REF`, checkout `detached HEAD` để cố định mã nguồn trong suốt 12 lượt sweep, lưu bảng băm tài nguyên đóng băng, và tự động khôi phục (`hydrate`) các lượt đã `complete` từ Google Drive nếu đang nối lại một phiên chạy bị ngắt.


In [17]:
import base64

def sha256_path(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_copy_file(src: Path, dst: Path) -> None:
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + f".{os.getpid()}.tmp")
    shutil.copy2(src, tmp)
    tmp.replace(dst)

def git_command(args, cwd=None) -> str:
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    if PRIVATE_REPO:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        assert token, "Thiếu Colab Secret GITHUB_TOKEN khi PRIVATE_REPO=True."
        auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
        env.update(
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.extraHeader",
            GIT_CONFIG_VALUE_0="Authorization: Basic " + auth,
        )
    result = subprocess.run(
        ["git", *args], cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Git thất bại: kiểm tra URL, quyền đọc hoặc commit SHA đã được push chưa.\n" + result.stderr
        )
    return result.stdout.strip()

if not REPO_DIR.exists():
    git_command(["clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)])
if not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(f"REPO_DIR tồn tại nhưng không phải Git checkout: {REPO_DIR}")
if not git_command(["ls-files"], REPO_DIR) and all(p.name == ".git" for p in REPO_DIR.iterdir()):
    git_command(["restore", "--source=HEAD", "--staged", "--worktree", "--", "."], REPO_DIR)
if git_command(["diff", "--name-only"], REPO_DIR) or git_command(["diff", "--cached", "--name-only"], REPO_DIR):
    raise RuntimeError("Checkout có thay đổi cục bộ; hãy chọn REPO_DIR sạch.")

git_command(["fetch", "--depth=1", "origin", REPO_REF], REPO_DIR)
git_command(["checkout", "--detach", "FETCH_HEAD"], REPO_DIR)
COMMIT = git_command(["rev-parse", "HEAD"], REPO_DIR)
assert len(COMMIT) == 40, f"Commit SHA không hợp lệ: {COMMIT}"

OUTPUT_DIR = OUTPUT_BASE / COMMIT[:12] / PROFILE_ID
RESULTS_DIR = OUTPUT_DIR / "results" / "phase2"
CKPT_OUT_DIR = OUTPUT_DIR / "checkpoints" / "phase2"
ENV_DIR = OUTPUT_DIR / "environment"
LOG_DIR = OUTPUT_DIR / "logs"
for folder in (RESULTS_DIR, CKPT_OUT_DIR, ENV_DIR, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DRIVE_SESSION_ROOT = DRIVE_BACKUP_BASE if (MOUNT_DRIVE and USE_DRIVE_BACKUP) else None
if DRIVE_SESSION_ROOT is not None:
    DRIVE_SESSION_ROOT.mkdir(parents=True, exist_ok=True)

required_repo_files = [
    "configs/phase2.yaml",
    "src/phase2/cli.py",
    "src/phase2/run.py",
    "src/phase2/models.py",
    "src/phase2/select_lr.py",
    "src/zs/ckpt.py",
    "data/clean/neuesc_100.csv",
    "data/clean/vsmec_train.csv",
    "data/clean/vsmec_dev.csv",
    "data/clean/vsmec_test.csv",
    "data/subsets_phase2/MANIFEST.json",
    "data/subsets_phase2/neuesc_f10_s0_ids.csv",
    "data/subsets_phase2/vsmec_f10_s0_ids.csv",
]
for rel in required_repo_files:
    assert (REPO_DIR / rel).is_file(), f"Thiếu file bắt buộc tại commit {COMMIT}: {rel}"

frozen_paths = [
    p
    for folder in ("data/clean", "data/splits", "data/subsets_phase2", "src/seq", "src/zs", "src/stats", "src/phase2")
    for p in sorted((REPO_DIR / folder).rglob("*"))
    if p.is_file() and "__pycache__" not in p.parts
]
frozen_paths += [
    REPO_DIR / rel
    for rel in ("configs/base.yaml", "configs/zs.yaml", "configs/seq.yaml", "configs/phase2.yaml", "src/common/constants.py")
]
frozen_snapshot = {p.relative_to(REPO_DIR).as_posix(): sha256_path(p) for p in frozen_paths}
frozen_file = ENV_DIR / "frozen_before.json"
if frozen_file.is_file():
    assert json.loads(frozen_file.read_text(encoding="utf-8")) == frozen_snapshot, "Tài nguyên đóng băng đã thay đổi!"
else:
    frozen_file.write_text(json.dumps(frozen_snapshot, sort_keys=True, indent=2) + "\n", encoding="utf-8")

# Khôi phục (hydrate) các lượt đã complete từ Drive khi mở lại phiên Colab mới
if DRIVE_SESSION_ROOT is not None and not (RESULTS_DIR / "phase2_runs.csv").is_file():
    drive_results = DRIVE_SESSION_ROOT
    drive_csv = drive_results / "phase2_runs.csv"
    if drive_csv.is_file():
        with drive_csv.open("r", encoding="utf-8", newline="") as handle:
            rows = list(csv.DictReader(handle))
        commits_in_csv = {r["git_commit"] for r in rows}
        assert commits_in_csv <= {COMMIT}, f"Backup trên Drive thuộc commit khác: {commits_in_csv} != {COMMIT}"
        assert all(r["run_mode"] == "sweep" and r["split"] == "dev" for r in rows), "Backup chứa dòng ngoài sweep/dev."
        verified_run_ids = set()
        for run_id in sorted({r["run_id"] for r in rows}):
            d_run = drive_results / "runs" / run_id
            d_prov = drive_results / "work" / run_id / "provenance.json"
            d_comp = d_run / "completion.json"
            if not (d_comp.is_file() and d_prov.is_file()):
                continue
            comp = json.loads(d_comp.read_text(encoding="utf-8"))
            prov = json.loads(d_prov.read_text(encoding="utf-8"))
            if comp.get("complete") is not True or comp.get("fingerprint") != prov.get("fingerprint"):
                continue
            artifacts = comp.get("artifacts", {})
            if not all((d_run / rel).resolve().is_file() and sha256_path((d_run / rel).resolve()) == digest
                       for rel, digest in artifacts.items()):
                continue
            atomic_copy_file(d_prov, RESULTS_DIR / "work" / run_id / "provenance.json")
            for item in sorted(d_run.rglob("*")):
                if item.is_file() and item.name != "completion.json" and item.suffix != ".tmp":
                    atomic_copy_file(item, RESULTS_DIR / "runs" / run_id / item.relative_to(d_run))
            atomic_copy_file(d_comp, RESULTS_DIR / "runs" / run_id / "completion.json")
            verified_run_ids.add(run_id)
        if verified_run_ids:
            kept_rows = [r for r in rows if r["run_id"] in verified_run_ids]
            with drive_csv.open("r", encoding="utf-8", newline="") as handle:
                fieldnames = csv.DictReader(handle).fieldnames
            tmp_csv = RESULTS_DIR / "phase2_runs.csv.tmp"
            with tmp_csv.open("w", encoding="utf-8", newline="") as handle:
                writer = csv.DictWriter(handle, fieldnames=fieldnames, lineterminator="\n")
                writer.writeheader()
                writer.writerows(kept_rows)
            tmp_csv.replace(RESULTS_DIR / "phase2_runs.csv")
            print(f"Hydrated {len(verified_run_ids)} completed sweep runs from Drive backup.")

print("Commit:", COMMIT)
print("Output dir:", OUTPUT_DIR)
print("Drive backup dir:", DRIVE_SESSION_ROOT)

Commit: edf4067751d72f84ad5e70485614f2d1e71b6d9c
Output dir: /content/task07_outputs/edf4067751d7/colab-task06-v1
Drive backup dir: /content/drive/MyDrive/Xdomain/Phase2/Task07


## 4. Kiểm tra hỗ trợ `venv`/`ensurepip` và dọn sạch venv tạo dở (nếu có)
Trên một số runtime Colab, gói hệ thống `pythonX.Y-venv` chưa được cài sẵn khiến lệnh `venv` tạo xong `bin/python` rồi mới lỗi ở bước `ensurepip`. Cell này kiểm tra và cài đúng `pythonX.Y-venv` trước, đồng thời xóa thư mục `VENV_DIR` tạo dở nếu thiếu `bin/pip`.


In [18]:
probe = subprocess.run([sys.executable, "-c", "import venv, ensurepip"], capture_output=True, text=True)
if probe.returncode != 0:
    py_mm = f"{sys.version_info.major}.{sys.version_info.minor}"
    apt_pkg = f"python{py_mm}-venv"
    print(f"Thiếu venv/ensurepip cho Python {py_mm}; đang cài gói hệ thống {apt_pkg}...")
    apt_upd = subprocess.run(["apt-get", "update", "-qq"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    apt_ins = subprocess.run(
        ["apt-get", "install", "-y", "-qq", apt_pkg], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    apt_log = (apt_upd.stdout or "") + "\n" + (apt_ins.stdout or "")
    (LOG_DIR / f"apt_venv_{SESSION_ID}.log").write_text(apt_log, encoding="utf-8")
    print(apt_log[-3000:])
    apt_upd.check_returncode()
    apt_ins.check_returncode()
    subprocess.run([sys.executable, "-c", "import venv, ensurepip"], check=True)

PYTHON = VENV_DIR / "bin" / "python"
PIP_BIN = VENV_DIR / "bin" / "pip"
if VENV_DIR.exists() and not (PYTHON.is_file() and PIP_BIN.is_file()):
    print("Phát hiện virtualenv tạo dở (thiếu python hoặc pip), đang xóa để tạo lại:", VENV_DIR)
    shutil.rmtree(VENV_DIR)
print("Venv support check: PASS")


Venv support check: PASS


## 5. Tạo virtualenv theo profile và cài bộ 12 package pin
Kế thừa PyTorch CUDA của Colab qua `--system-site-packages`, cài đúng 12 package pin bằng `PYTHON -m pip`, và **ghi ngay** `install.log`, `pip-freeze.txt`, `pip_check.txt` (`pip check` chạy với `check=False` vì gói nền Colab luôn báo xung đột ngoài phạm vi pipeline).


In [19]:
if not PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV_DIR)], check=True)
assert PYTHON.is_file() and PIP_BIN.is_file(), f"Tạo virtualenv thất bại: {VENV_DIR}"

pins_list = [f"{name}=={ver}" for name, ver in PROFILES[PROFILE_ID]["pins"].items()]
install = subprocess.run(
    [str(PYTHON), "-m", "pip", "install", *pins_list],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
for install_path in (LOG_DIR / f"install_{SESSION_ID}.log", LOG_DIR / "install.log"):
    install_path.write_text(install.stdout, encoding="utf-8")
print(install.stdout[-4500:])
install.check_returncode()

freeze_out = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True)
for freeze_path in (ENV_DIR / f"pip-freeze_{SESSION_ID}.txt", ENV_DIR / "pip-freeze.txt"):
    freeze_path.write_text(freeze_out, encoding="utf-8")

pip_chk = subprocess.run(
    [str(PYTHON), "-m", "pip", "check"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False
)
pip_chk_text = (pip_chk.stdout or "") + f"\n[pip_check_exit_code]={pip_chk.returncode}\n"
for chk_path in (ENV_DIR / f"pip_check_{SESSION_ID}.txt", ENV_DIR / "pip_check.txt"):
    chk_path.write_text(pip_chk_text, encoding="utf-8")

RUN_ENV = os.environ.copy()
RUN_ENV.update(
    PYTHONUNBUFFERED="1",
    TOKENIZERS_PARALLELISM="false",
    HF_HUB_DISABLE_TELEMETRY="1",
)
RUN_ENV.pop("HF_HUB_OFFLINE", None)
RUN_ENV.pop("TRANSFORMERS_OFFLINE", None)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        RUN_ENV["HF_TOKEN"] = hf_token.strip()
        print("Loaded optional HF_TOKEN from Colab Secrets (redacted).")
except Exception:
    pass

def run_python(source: str) -> str:
    result = subprocess.run(
        [str(PYTHON), "-u", "-c", source],
        cwd=REPO_DIR,
        env=RUN_ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    result.check_returncode()
    return result.stdout

print(f"Installed {len(pins_list)} pinned packages into {VENV_DIR} (pip check exit={pip_chk.returncode}).")



Installed 12 pinned packages into /content/phase2-task07-colab-task06-v1-venv (pip check exit=1).


## 6. Preflight môi trường & kiểm tra forward/loss BF16 bằng Python của venv
Chạy trong subprocess `PYTHON` riêng (giải phóng sạch VRAM ngay khi kết thúc): ghi `environment_<SESSION_ID>.json` **trước khi `assert`** để luôn giữ được bằng chứng môi trường nếu Colab nâng phiên bản, kiểm tra toàn bộ 12 pin và chạy thử 1 bước forward/loss BF16 hữu hạn của `DFT` (đồng thời tải sẵn cache `uitnlp/visobert` trước sweep).


In [20]:
preflight_env_script = r'''
import importlib.metadata as metadata
import json, platform, subprocess, sys
from pathlib import Path
import numpy, pandas, scipy, sklearn, torch, transformers, yaml
from src.phase2.models import build_phase2_model
from src.phase2.protocol import load_phase2_config
from src.phase2.run import atomic_json, run_phase2_experiment

ctx = json.loads(__CTX_JSON__)
profile_spec = ctx["profile_spec"]
env_dir = Path(ctx["env_dir"])

actual_pins = {}
mismatches = []
for pkg, exp_ver in profile_spec["pins"].items():
    try:
        got_ver = metadata.version(pkg)
    except metadata.PackageNotFoundError:
        got_ver = None
    actual_pins[pkg] = got_ver
    if got_ver != exp_ver:
        mismatches.append(f"{pkg}: expected {exp_ver}, got {got_ver}")

py_ver = platform.python_version()
if py_ver != profile_spec["python"]:
    mismatches.append(f"python: expected {profile_spec['python']}, got {py_ver}")
if torch.__version__ != profile_spec["torch"]:
    mismatches.append(f"torch: expected {profile_spec['torch']}, got {torch.__version__}")
if torch.version.cuda != profile_spec["cuda"]:
    mismatches.append(f"cuda: expected {profile_spec['cuda']}, got {torch.version.cuda}")
if not torch.cuda.is_available():
    mismatches.append("torch.cuda.is_available() is False")
elif not torch.cuda.is_bf16_supported():
    mismatches.append("torch.cuda.is_bf16_supported() is False")

try:
    smi_out = subprocess.check_output(["nvidia-smi"], text=True)
except Exception as exc:
    smi_out = f"unavailable: {exc}"

snapshot = {
    "profile_id": ctx["profile_id"],
    "session_id": ctx["session_id"],
    "commit": ctx["commit"],
    "python": py_ver,
    "sys_executable": sys.executable,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "cuda_available": torch.cuda.is_available(),
    "bf16_supported": bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vram_total_bytes": int(torch.cuda.get_device_properties(0).total_memory) if torch.cuda.is_available() else 0,
    "nvidia_smi": smi_out,
    "pinned_packages_expected": profile_spec["pins"],
    "pinned_packages_actual": actual_pins,
    "module_files": {
        "torch": getattr(torch, "__file__", None),
        "transformers": getattr(transformers, "__file__", None),
        "numpy": getattr(numpy, "__file__", None),
        "pandas": getattr(pandas, "__file__", None),
        "sklearn": getattr(sklearn, "__file__", None),
        "scipy": getattr(scipy, "__file__", None),
        "yaml": getattr(yaml, "__file__", None),
    },
    "preflight_mismatches": mismatches,
    "preflight_status": "FAIL" if mismatches else "ENV_PASS",
}
session_env_file = env_dir / f"environment_{ctx['session_id']}.json"
atomic_json(session_env_file, snapshot)
atomic_json(env_dir / "environment.json", snapshot)

if mismatches:
    raise AssertionError(
        "Profile preflight FAILED (đã lưu chi tiết tại "
        + str(session_env_file)
        + "; không âm thầm dùng tiếp profile cũ): "
        + "; ".join(mismatches)
    )

cfg = load_phase2_config("configs/phase2.yaml")
model, tok, dft_prov = build_phase2_model("dft", "SA", "ViSoBERT", 0, 10, cfg)
model = model.to("cuda").train()
enc = tok(["kiểm tra tiền huấn luyện một", "kiểm tra tiền huấn luyện hai"], padding=True, truncation=True, max_length=32, return_tensors="pt")
batch = {k: v.to("cuda") for k, v in enc.items()}
batch["labels"] = torch.tensor([0, 2], dtype=torch.long, device="cuda")
with torch.autocast("cuda", dtype=torch.bfloat16):
    out = model(**batch)
assert out.logits.shape == (2, 3) and bool(torch.isfinite(out.loss)), "DFT BF16 forward/loss preflight failed!"
del out, batch, enc, model, tok
torch.cuda.empty_cache()

snapshot["dft_pretrained_preflight"] = {"status": "PASS", "provenance": dft_prov}
atomic_json(session_env_file, snapshot)
atomic_json(env_dir / "environment.json", snapshot)
print("GPU:", snapshot["gpu"])
print("Environment & DFT BF16 preflight: PASS")
'''

ctx_payload = json.dumps({
    "profile_id": PROFILE_ID,
    "profile_spec": PROFILES[PROFILE_ID],
    "session_id": SESSION_ID,
    "commit": COMMIT,
    "env_dir": str(ENV_DIR),
})
run_python(preflight_env_script.replace("__CTX_JSON__", repr(ctx_payload)))

/content/phase2-task07-colab-task06-v1-venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
I0000 00:00:1795779105.473944   17351 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795779105.546212   17351 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795779108.058124   17351 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-

"/content/phase2-task07-colab-task06-v1-venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).\n  from pandas.core import (\nWARNING: All log messages before absl::InitializeLog() is called are written to STDERR\nI0000 00:00:1795779105.473944   17351 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.\nI0000 00:00:1795779105.546212   17351 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.\nTo enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.\nWARNING: All log messages before absl::InitializeLog() is called ar

## 7. Stage 2 checkpoint nguồn Stage 1 (`seed 0`), kiểm tra dữ liệu/giao thức & xác minh `Seq` (`SA` + `SB`)
Copy thật (`shutil.copytree`, không dùng symlink) 2 checkpoint nguồn `ViSoBERT` `seed=0` (`vsmec` và `neu3`) từ Drive về `/content/phase2_source_ckpts` (giữ nguyên tên thư mục gốc và chép kèm `stage1_ckpt_hash.json` / `neu3_ckpt_hash.json` nếu có), sau đó xác minh cách ly `test` và chạy thử 1 bước forward/loss BF16 trên GPU cho **cả hai nhánh `Seq SA` và `Seq SB`** trước khi bắt đầu sweep.


In [21]:
stage_and_data_script = r'''
import json, shutil, time
from pathlib import Path
import torch
from src.common.hashing import sha256_file
from src.phase2.data import load_phase2_prepared_data
from src.phase2.models import build_phase2_model, file_hashes
from src.phase2.protocol import load_phase2_config
from src.phase2.run import atomic_json
from src.plm.model_ckpt import _find_encoder_weight_file
from src.zs.ckpt import locate_checkpoint

ctx = json.loads(__CTX_JSON__)
stage_root = Path(ctx["stage_root"])
assert "sweep" not in str(stage_root).lower() and "smoke" not in str(stage_root).lower()
stage_root.mkdir(parents=True, exist_ok=True)

IGNORE_NAMES = {".ipynb_checkpoints", ".DS_Store", "__pycache__", "Thumbs.db"}

def ignore_junk(_dir, names):
    return {n for n in names if n in IGNORE_NAMES or n.endswith(".tmp")}

def find_matching_external_manifests(checkpoint: Path, search_root: Path, run_name: str, alias: str, domain: str):
    filename = "stage1_ckpt_hash.json" if domain == "vsmec" else "neu3_ckpt_hash.json"
    dirs = {search_root, search_root.parent, checkpoint, checkpoint.parent, checkpoint.parent.parent}
    found = []
    for d in sorted(dirs, key=str):
        for candidate in (d / filename, d / "results" / filename):
            if not candidate.is_file():
                continue
            payload = json.loads(candidate.read_text(encoding="utf-8"))
            if not isinstance(payload, dict):
                continue
            for key, entry in payload.items():
                if not isinstance(entry, dict):
                    continue
                parts = str(entry.get("path", "")).replace("\\", "/").split("/")
                if key == run_name or run_name in parts:
                    found.append(("root", filename, key, entry))
                elif (key == alias and run_name in candidate.parent.parts) or (
                    key == alias and Path(str(entry.get("path", ""))).resolve() == checkpoint.resolve()
                ):
                    found.append(("run", filename, key, entry))
    return found

staged_summary = {}
root_manifests_to_write = {}

for domain, src_root_str in ctx["search_roots"].items():
    src_root = Path(src_root_str)
    src_ckpt = locate_checkpoint(src_root, "ViSoBERT", domain, 0)
    run_folder = src_ckpt.parent if src_ckpt.name == "ViSoBERT" else src_ckpt
    run_name = run_folder.name
    dst_run_folder = stage_root / run_name

    src_enc_sha = sha256_file(_find_encoder_weight_file(src_ckpt / "encoder"))
    src_head_sha = sha256_file(src_ckpt / "head.pt")

    need_copy = True
    if dst_run_folder.is_dir():
        dst_ckpt_guess = (dst_run_folder / "ViSoBERT") if src_ckpt.name == "ViSoBERT" else dst_run_folder
        if (dst_ckpt_guess / "head.pt").is_file() and (dst_ckpt_guess / "encoder" / "config.json").is_file():
            try:
                if (
                    sha256_file(_find_encoder_weight_file(dst_ckpt_guess / "encoder")) == src_enc_sha
                    and sha256_file(dst_ckpt_guess / "head.pt") == src_head_sha
                ):
                    need_copy = False
            except Exception:
                need_copy = True
        if need_copy:
            shutil.rmtree(dst_run_folder)

    if need_copy:
        shutil.copytree(run_folder, dst_run_folder, symlinks=False, ignore=ignore_junk)

    for scope, filename, key, entry in find_matching_external_manifests(src_ckpt, src_root, run_name, "ViSoBERT", domain):
        if scope == "run":
            target_manifest = dst_run_folder / "results" / filename
            if not target_manifest.is_file() and not (dst_run_folder / filename).is_file():
                atomic_json(target_manifest, {key: entry})
        else:
            root_manifests_to_write.setdefault(filename, {})[key] = entry

for filename, payload in root_manifests_to_write.items():
    target_manifest = stage_root / filename
    merged = json.loads(target_manifest.read_text(encoding="utf-8")) if target_manifest.is_file() else {}
    merged.update(payload)
    atomic_json(target_manifest, merged)

for domain, src_root_str in ctx["search_roots"].items():
    src_ckpt = locate_checkpoint(Path(src_root_str), "ViSoBERT", domain, 0)
    staged_ckpt = locate_checkpoint(stage_root, "ViSoBERT", domain, 0)
    assert not staged_ckpt.is_symlink(), f"Staged checkpoint không được là symlink: {staged_ckpt}"
    assert sha256_file(_find_encoder_weight_file(staged_ckpt / "encoder")) == sha256_file(
        _find_encoder_weight_file(src_ckpt / "encoder")
    ), f"Encoder SHA-256 mismatch sau khi stage ({domain})"
    assert sha256_file(staged_ckpt / "head.pt") == sha256_file(src_ckpt / "head.pt"), (
        f"Head SHA-256 mismatch sau khi stage ({domain})"
    )
    staged_summary[domain] = {
        "drive_source": str(src_ckpt),
        "staged_path": str(staged_ckpt),
        "encoder_sha256": sha256_file(_find_encoder_weight_file(staged_ckpt / "encoder")),
        "head_sha256": sha256_file(staged_ckpt / "head.pt"),
        "n_files": len(file_hashes(staged_ckpt)),
    }
    print(f"Staged & verified ({domain}): {staged_ckpt} | encoder_sha256={staged_summary[domain]['encoder_sha256'][:12]}...")

cfg = load_phase2_config("configs/phase2.yaml")
assert cfg["protocol_version"] == "phase2-v1"
assert cfg["sweeps"]["fraction"] == 10 and cfg["sweeps"]["seed"] == 0
assert cfg["sweeps"]["dft_lr_grid"] == [2e-5, 3e-5, 5e-5]
assert cfg["sweeps"]["seq_lr_grid"] == [1e-5, 2e-5, 3e-5]
assert cfg["training"]["strict_precision"] is True and cfg["training"]["precision"] == "bf16"

expected_shapes = {"SA": (2305, 3305, 3), "SB": (554, 686, 7)}
data_prov = {}
seq_prov = {}

for direction, (exp_train, exp_dev, num_labels) in expected_shapes.items():
    prepared = load_phase2_prepared_data(direction, 10, 0, "ViSoBERT", cfg, run_mode="sweep")
    assert len(prepared["train_df"]) == exp_train and len(prepared["dev_df"]) == exp_dev, direction
    assert prepared["test_df"] is None and prepared["test_texts"] is None and prepared["test_labels"] is None, direction
    assert prepared["provenance"]["test_integrity_checked"] is False, direction
    data_prov[direction] = prepared["provenance"]

    model, tok, prov = build_phase2_model(
        "seq", direction, "ViSoBERT", 0, 10, cfg, source_ckpt_root=stage_root
    )
    model = model.to("cuda").train()
    enc = tok(prepared["train_texts"][:2], padding=True, truncation=True, max_length=cfg["training"]["max_length"], return_tensors="pt")
    batch = {k: v.to("cuda") for k, v in enc.items()}
    batch["labels"] = torch.tensor(prepared["train_labels"][:2], dtype=torch.long, device="cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16):
        out = model(**batch)
    assert out.logits.shape == (2, num_labels) and bool(torch.isfinite(out.loss)), f"Seq {direction} BF16 preflight failed!"
    del out, batch, enc, model, tok
    torch.cuda.empty_cache()
    seq_prov[direction] = prov
    print(f"Data + Seq BF16 preflight ({direction}): PASS | train={exp_train}, dev={exp_dev}, labels={num_labels}")

env_dir = Path(ctx["env_dir"])
session_env_file = env_dir / f"environment_{ctx['session_id']}.json"
snapshot = json.loads(session_env_file.read_text(encoding="utf-8"))
snapshot.update({
    "preflight_status": "PASS",
    "setup_seconds_before_sweep": round(time.perf_counter() - float(ctx["t0_setup"]), 3),
    "data_provenance": data_prov,
    "source_ckpt_provenance": seq_prov,
    "staged_source_checkpoints": staged_summary,
})
atomic_json(session_env_file, snapshot)
atomic_json(env_dir / "environment.json", snapshot)
print("Full Step 7 preflight & provenance recording: PASS")
'''

stage_ctx = json.dumps({
    "stage_root": str(SOURCE_CKPT_STAGE_DIR),
    "search_roots": {k: str(v) for k, v in CKPT_SEARCH_ROOTS.items()},
    "env_dir": str(ENV_DIR),
    "session_id": SESSION_ID,
    "t0_setup": T0_SETUP,
})
run_python(stage_and_data_script.replace("__CTX_JSON__", repr(stage_ctx)))

/content/phase2-task07-colab-task06-v1-venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
Staged & verified (vsmec): /content/phase2_source_ckpts/ViSoBERT_Stage1-base-emokeep-bf16_s0_lr5e-5 | encoder_sha256=3cf5ebc839dc...
Staged & verified (neu3): /content/phase2_source_ckpts/ViSoBERT_NEU3-base-emokeep-bf16_s0_lr3e-5 | encoder_sha256=da379459f87d...
I0000 00:00:1795779140.795106   17503 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795779140.866639   17503 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 

"/content/phase2-task07-colab-task06-v1-venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).\n  from pandas.core import (\nStaged & verified (vsmec): /content/phase2_source_ckpts/ViSoBERT_Stage1-base-emokeep-bf16_s0_lr5e-5 | encoder_sha256=3cf5ebc839dc...\nStaged & verified (neu3): /content/phase2_source_ckpts/ViSoBERT_NEU3-base-emokeep-bf16_s0_lr3e-5 | encoder_sha256=da379459f87d...\nWARNING: All log messages before absl::InitializeLog() is called are written to STDERR\nI0000 00:00:1795779140.795106   17503 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.\nI0000 00:00:1795779140.866639   17503 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CP

## 8. Thực thi 12 lượt sweep (`dev`-only) và khóa learning rate (`src.phase2.select_lr`)
- **Cell 8a:** Chạy tuần tự 12 lượt theo thứ tự gom theo hướng (`SA dft → SA seq → SB dft → SB seq`) trong subprocess `PYTHON`. Không bật ngắt nhân tạo `VERIFY_RESUME`. Ngay sau mỗi lượt hoàn tất (`complete` hoặc `SKIP`):
  1. Xóa riêng `work/<run_id>/resume.pt` nếu `CLEAN_COMPLETED_RESUME_PT=True` để tiết kiệm ~13,2 GB trên `/content`, nhưng **luôn giữ lại `work/<run_id>/provenance.json`**.
  2. Đồng bộ nguyên tử `work/<run_id>/provenance.json`, `runs/<run_id>/` và `phase2_runs.csv` sang Google Drive (`USE_DRIVE_BACKUP=True`).
- **Cell 8b:** Gọi `PYTHON -m src.phase2.select_lr` khi đủ 12 lượt để sinh `selection_p1_lr.json` và `p1_lr_selection.csv`.


In [22]:
SWEEP_JOBS = [
    ("SA", "dft", 2.0e-5),
    ("SA", "dft", 3.0e-5),
    ("SA", "dft", 5.0e-5),
    ("SA", "seq", 1.0e-5),
    ("SA", "seq", 2.0e-5),
    ("SA", "seq", 3.0e-5),
    ("SB", "dft", 2.0e-5),
    ("SB", "dft", 3.0e-5),
    ("SB", "dft", 5.0e-5),
    ("SB", "seq", 1.0e-5),
    ("SB", "seq", 2.0e-5),
    ("SB", "seq", 3.0e-5),
]

TRAIN_LOG_PATH = LOG_DIR / f"train_{SESSION_ID}.log"
COMMANDS_RECORD = []

def stop_process_group(proc: subprocess.Popen) -> None:
    if proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGINT)
            proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait(timeout=30)
        except ProcessLookupError:
            pass

def sync_completed_run_to_drive(run_id: str) -> None:
    if DRIVE_SESSION_ROOT is None:
        return
    d_res = DRIVE_SESSION_ROOT
    local_prov = RESULTS_DIR / "work" / run_id / "provenance.json"
    local_run_dir = RESULTS_DIR / "runs" / run_id
    atomic_copy_file(local_prov, d_res / "work" / run_id / "provenance.json")
    for item in sorted(local_run_dir.rglob("*")):
        if item.is_file() and item.name != "completion.json" and item.suffix != ".tmp":
            atomic_copy_file(item, d_res / "runs" / run_id / item.relative_to(local_run_dir))
    atomic_copy_file(local_run_dir / "completion.json", d_res / "runs" / run_id / "completion.json")
    atomic_copy_file(RESULTS_DIR / "phase2_runs.csv", d_res / "phase2_runs.csv")

for idx, (direction, method, lr) in enumerate(SWEEP_JOBS, start=1):
    cmd = [
        str(PYTHON), "-u", "-m", "src.phase2.cli",
        "--config", "configs/phase2.yaml",
        "--backbone", "ViSoBERT",
        "--direction", direction,
        "--method", method,
        "--fraction", "10",
        "--seed", "0",
        "--lr", f"{lr:g}",
        "--run-mode", "sweep",
        "--results-dir", str(RESULTS_DIR),
        "--ckpt-out-dir", str(CKPT_OUT_DIR),
    ]
    if method == "seq":
        cmd += ["--source-ckpt-root", str(SOURCE_CKPT_STAGE_DIR)]

    COMMANDS_RECORD.append({"index": idx, "direction": direction, "method": method, "lr": lr, "argv": cmd})
    (LOG_DIR / f"commands_{SESSION_ID}.json").write_text(
        json.dumps(COMMANDS_RECORD, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )

    with TRAIN_LOG_PATH.open("a", encoding="utf-8") as log_handle:
        start_offset = log_handle.tell()
        header = (
            f"\n=== [{idx}/12] {direction}/{method}/lr={lr:g} start "
            f"{time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())} ===\n"
        )
        log_handle.write(header)
        log_handle.flush()

        t_job = time.perf_counter()
        proc = subprocess.Popen(
            cmd,
            cwd=REPO_DIR,
            env=RUN_ENV,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        last_heartbeat = time.monotonic()
        try:
            while proc.poll() is None:
                time.sleep(3)
                if time.monotonic() - last_heartbeat >= 30:
                    last_heartbeat = time.monotonic()
                    tail_txt = TRAIN_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-500:]
                    last_line = [ln for ln in tail_txt.splitlines() if ln.strip()][-1:] or ["..."]
                    print(f"  [{idx}/12 running {round(time.perf_counter() - t_job)}s] {last_line[0]}")
        except BaseException:
            stop_process_group(proc)
            raise

    with TRAIN_LOG_PATH.open("r", encoding="utf-8", errors="replace") as read_handle:
        read_handle.seek(start_offset)
        segment = read_handle.read()

    if proc.returncode != 0:
        print(segment[-6000:])
        raise RuntimeError(f"Lượt [{idx}/12] ({direction}/{method}/lr={lr:g}) thất bại (exit={proc.returncode}).")

    matches = re.findall(r"^([A-Za-z0-9._\-]+): (complete|SKIP)\s*$", segment, flags=re.MULTILINE)
    assert matches, f"Không tìm thấy dòng trạng thái '<run_id>: complete|SKIP' ở lượt [{idx}/12].\n{segment[-1500:]}"
    run_id, run_status = matches[-1]

    comp_path = RESULTS_DIR / "runs" / run_id / "completion.json"
    prov_path = RESULTS_DIR / "work" / run_id / "provenance.json"
    assert comp_path.is_file() and json.loads(comp_path.read_text(encoding="utf-8")).get("complete") is True, (
        f"completion.json chưa hợp lệ cho {run_id}"
    )
    assert prov_path.is_file(), f"Thiếu provenance.json cho {run_id} (bắt buộc cho select_lr.py)!"

    if CLEAN_COMPLETED_RESUME_PT:
        resume_pt = RESULTS_DIR / "work" / run_id / "resume.pt"
        if resume_pt.is_file():
            resume_pt.unlink()
        assert prov_path.is_file(), f"provenance.json bị mất sau khi dọn resume.pt ({run_id})!"

    sync_completed_run_to_drive(run_id)

    metrics_obj = json.loads((RESULTS_DIR / "runs" / run_id / "metrics.json").read_text(encoding="utf-8"))
    cost_obj = json.loads((RESULTS_DIR / "runs" / run_id / "cost.json").read_text(encoding="utf-8"))
    print(
        f"[{idx:02d}/12] {run_id} | {run_status} | "
        f"dev_score={metrics_obj['dev_selection_score']:.4f} | "
        f"best_ep={metrics_obj['best_epoch']} | steps={cost_obj['actual_update_steps']} | "
        f"train_s={cost_obj['train_seconds']:.1f}s"
    )

print("Completed all 12 Task07 sweep runs.")

[01/12] ViSoBERT_SA_dft_f10_s0_lr2e-05_p2v1_had5c9c35 | SKIP | dev_score=0.7037 | best_ep=4.0 | steps=438 | train_s=86.6s
[02/12] ViSoBERT_SA_dft_f10_s0_lr3e-05_p2v1_h09960405 | SKIP | dev_score=0.7166 | best_ep=8.0 | steps=730 | train_s=144.3s
[03/12] ViSoBERT_SA_dft_f10_s0_lr5e-05_p2v1_hd37168c0 | SKIP | dev_score=0.7095 | best_ep=8.0 | steps=730 | train_s=145.5s
[04/12] ViSoBERT_SA_seq_f10_s0_lr1e-05_p2v1_h42b0806d | SKIP | dev_score=0.6440 | best_ep=7.0 | steps=657 | train_s=142.0s
[05/12] ViSoBERT_SA_seq_f10_s0_lr2e-05_p2v1_had5c9c35 | SKIP | dev_score=0.6592 | best_ep=8.0 | steps=730 | train_s=157.7s
[06/12] ViSoBERT_SA_seq_f10_s0_lr3e-05_p2v1_h09960405 | SKIP | dev_score=0.6294 | best_ep=4.0 | steps=438 | train_s=94.5s
[07/12] ViSoBERT_SB_dft_f10_s0_lr2e-05_p2v1_had5c9c35 | SKIP | dev_score=0.5960 | best_ep=5.0 | steps=126 | train_s=41.5s
[08/12] ViSoBERT_SB_dft_f10_s0_lr3e-05_p2v1_h09960405 | SKIP | dev_score=0.5714 | best_ep=4.0 | steps=108 | train_s=41.4s
[09/12] ViSoBERT_SB_

In [23]:
select_cmd = [
    str(PYTHON), "-u", "-m", "src.phase2.select_lr",
    "--results-dir", str(RESULTS_DIR),
    "--config", "configs/phase2.yaml",
]
select_res = subprocess.run(
    select_cmd, cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
(LOG_DIR / "select_lr.log").write_text(select_res.stdout, encoding="utf-8")
if select_res.stdout.strip():
    print(select_res.stdout)
select_res.check_returncode()

selection_json_path = RESULTS_DIR / "selection_p1_lr.json"
selection_csv_path = RESULTS_DIR / "p1_lr_selection.csv"
assert selection_json_path.is_file() and selection_csv_path.is_file(), "Thiếu file đầu ra của select_lr!"

selection_payload = json.loads(selection_json_path.read_text(encoding="utf-8"))
print("=== p1_lr_selection.csv ===")
print(selection_csv_path.read_text(encoding="utf-8"))
print("=== selection_p1_lr.json ===")
print(json.dumps(selection_payload, ensure_ascii=False, indent=2))

edge_hits = [k for k, rec in selection_payload["selected"].items() if rec.get("grid_edge")]
if edge_hits:
    print(
        "WARNING [GRID EDGE]: LR chiến thắng rơi vào biên lưới ở:",
        ", ".join(edge_hits),
        "-> Ghi nhận để nhóm xem xét theo giao thức (KHÔNG tự chạy thêm LR ngoài lưới vào cùng RESULTS_DIR).",
    )

if DRIVE_SESSION_ROOT is not None:
    atomic_copy_file(selection_json_path, DRIVE_SESSION_ROOT / "selection_p1_lr.json")
    atomic_copy_file(selection_csv_path, DRIVE_SESSION_ROOT / "p1_lr_selection.csv")
    print("Synced selection_p1_lr.json & p1_lr_selection.csv to Drive:", DRIVE_SESSION_ROOT)

/content/phase2-task07-colab-task06-v1-venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
/content/xdomain-task07/src/phase2/select_lr.py:69: UserWarning: SB/dft: selected LR 2e-05 is at grid edge
  warnings.warn(f"{direction}/{method}: selected LR {winner.lr:g} is at grid edge", UserWarning)
/content/xdomain-task07/src/phase2/select_lr.py:69: UserWarning: SB/seq: selected LR 3e-05 is at grid edge
  warnings.warn(f"{direction}/{method}: selected LR {winner.lr:g} is at grid edge", UserWarning)

=== p1_lr_selection.csv ===
backbone,direction,method,lr,dev_selection_score,data_hash,ids_sha256,run_id,grid_edge
ViSoBERT,SA,dft,3e-05,0.7166298007253648,1650177a9b31d43895f5138d6d504402b86c139c89cd241442d777483163af97,14f5fba3a6823278a13894e9e4dd017fba9461c373cbdfb515c389e978d3a218,ViSoBERT_SA_dft_f10_s0_lr3e-05_p2v1_h09960405,False
ViSoBERT,S

## 9. Kiểm tra `SKIP` và nghiệm thu độc lập toàn bộ 12 lượt sweep
Kiểm tra gọi lại CLI trả `SKIP` cho cả 4 nhánh chiến thắng (`SA/dft`, `SA/seq`, `SB/dft`, `SB/seq`), xác minh đủ 12 `completion.json`, 12 `provenance.json`, đúng **`12` `run_id` / `18` dòng `dev`** trong `phase2_runs.csv` (`6` dòng `SA` `V3-NEU` + `12` dòng `SB` gồm `V5-VSMEC` và `V7-VSMEC`), không có `predictions_test.csv`, xác suất hợp lệ và tài nguyên đóng băng nguyên vẹn.


In [24]:
selection_payload = json.loads((RESULTS_DIR / "selection_p1_lr.json").read_text(encoding="utf-8"))
skip_logs = []
for key in ("ViSoBERT/SA/dft", "ViSoBERT/SA/seq", "ViSoBERT/SB/dft", "ViSoBERT/SB/seq"):
    rec = selection_payload["selected"][key]
    cmd = [
        str(PYTHON), "-u", "-m", "src.phase2.cli",
        "--config", "configs/phase2.yaml",
        "--backbone", "ViSoBERT",
        "--direction", rec["direction"],
        "--method", rec["method"],
        "--fraction", "10",
        "--seed", "0",
        "--lr", f"{float(rec['lr']):g}",
        "--run-mode", "sweep",
        "--results-dir", str(RESULTS_DIR),
        "--ckpt-out-dir", str(CKPT_OUT_DIR),
    ]
    if rec["method"] == "seq":
        cmd += ["--source-ckpt-root", str(SOURCE_CKPT_STAGE_DIR)]
    res = subprocess.run(cmd, cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    skip_logs.append(f"$ {' '.join(cmd)}\n{res.stdout}")
    print(res.stdout.strip())
    res.check_returncode()
    assert f"{rec['run_id']}: SKIP" in res.stdout, f"Lượt {rec['run_id']} không SKIP khi gọi lại!"

(LOG_DIR / "skip_check.log").write_text("\n".join(skip_logs) + "\n", encoding="utf-8")

verify_script = r'''
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
from src.common.hashing import sha256_file
from src.common.schema import ZS_PRED_COLUMNS
from src.phase2.protocol import canonical_digest, compute_dev_selection_score, load_phase2_config

ctx = json.loads(__CTX_JSON__)
output_dir = Path(ctx["output_dir"])
results_dir = Path(ctx["results_dir"])
ckpt_out_dir = Path(ctx["ckpt_out_dir"])
env_dir = Path(ctx["env_dir"])
commit = ctx["commit"]

cfg = load_phase2_config("configs/phase2.yaml")

assert not list(output_dir.rglob("predictions_test.csv")), "Phát hiện predictions_test.csv trong sweep!"
assert not list(ckpt_out_dir.rglob("head.pt")), "Sweep không được xuất checkpoint main head.pt!"

manifests = sorted((results_dir / "runs").glob("*/completion.json"))
assert len(manifests) == 12, f"Cần đúng 12 completion.json, tìm thấy {len(manifests)}"

prov_files = sorted((results_dir / "work").glob("*/provenance.json"))
assert len(prov_files) == 12, f"Cần đúng 12 provenance.json, tìm thấy {len(prov_files)}"

code_digests = set()
seen_combos = set()
required_artifacts = {"config.yaml", "metadata.json", "metrics.json", "predictions_dev.csv", "cost.json"}

for manifest_path in manifests:
    run_dir = manifest_path.parent
    run_id = run_dir.name
    comp = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert comp.get("complete") is True and comp.get("ckpt_path") is None, run_id
    artifacts = comp.get("artifacts", {})
    assert required_artifacts.issubset(artifacts), run_id
    for rel, digest in artifacts.items():
        target = (run_dir / rel).resolve()
        assert target.is_file() and sha256_file(target) == digest, f"Hash lệch tại {run_id}/{rel}"

    prov_path = results_dir / "work" / run_id / "provenance.json"
    assert prov_path.is_file(), f"Thiếu provenance.json cho {run_id}"
    prov = json.loads(prov_path.read_text(encoding="utf-8"))
    assert prov.get("fingerprint") == comp.get("fingerprint"), f"Fingerprint lệch cho {run_id}"
    code_digests.add(canonical_digest(prov["source_code_sha256"]))

    meta = json.loads((run_dir / "metadata.json").read_text(encoding="utf-8"))
    assert meta["run_id"] == run_id and meta["run_mode"] == "sweep", run_id
    assert meta["backbone"] == "ViSoBERT" and meta["fraction"] == 10 and meta["seed"] == 0, run_id
    assert meta["max_examples"] is None and meta["git_commit"] == commit, run_id
    seen_combos.add((meta["direction"], meta["method"], float(meta["lr"])))

    metrics = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    cost = json.loads((run_dir / "cost.json").read_text(encoding="utf-8"))
    assert metrics["test_metrics"] is None and len(metrics["history"]) >= 1, run_id
    assert all(math.isfinite(h["train_loss"]) for h in metrics["history"]), run_id
    assert cost["effective_precision"] == "bf16" and cost["n_gpus"] == 1, run_id
    assert cost["train_seconds"] > 0 and cost["peak_vram_mb"] > 0, run_id
    assert cost["total_params"] == cost["trainable_params"] > 0, run_id

    pred = pd.read_csv(run_dir / "predictions_dev.csv")
    assert tuple(pred.columns) == ZS_PRED_COLUMNS and pred["id"].is_unique, run_id
    p3 = pred[["p3_neg", "p3_neu", "p3_pos"]].to_numpy(float)
    assert np.isfinite(p3).all() and (p3 >= 0).all() and np.allclose(p3.sum(axis=1), 1.0, atol=1e-6), run_id
    if meta["direction"] == "SA":
        assert len(pred) == 3305, run_id
        recomputed = compute_dev_selection_score(pred["y_true3"], p3.argmax(axis=1), "neuesc")
    else:
        assert len(pred) == 686, run_id
        p7 = pred[[f"p7_{i}" for i in range(7)]].to_numpy(float)
        assert np.isfinite(p7).all() and (p7 >= 0).all() and np.allclose(p7.sum(axis=1), 1.0, atol=1e-6), run_id
        recomputed = compute_dev_selection_score(pred["y_true7"], p7.argmax(axis=1), "vsmec")
    assert np.isclose(recomputed, metrics["dev_selection_score"], rtol=0, atol=1e-12), run_id

expected_combos = {
    (d, m, float(lr))
    for d in ("SA", "SB")
    for m, grid in (("dft", cfg["sweeps"]["dft_lr_grid"]), ("seq", cfg["sweeps"]["seq_lr_grid"]))
    for lr in grid
}
assert seen_combos == expected_combos, f"Tập cấu hình sweep không khớp lưới khóa: {seen_combos}"
assert len(code_digests) == 1, "source_code_sha256 không đồng nhất giữa 12 lượt!"

rows = pd.read_csv(results_dir / "phase2_runs.csv", keep_default_na=False)
assert rows["run_id"].nunique() == 12, f"Cần 12 run_id duy nhất, có {rows['run_id'].nunique()}"
assert len(rows) == 18, f"Cần đúng 18 dòng dev (6 SA + 12 SB), có {len(rows)}"
assert rows["split"].eq("dev").all() and rows["run_mode"].eq("sweep").all()
assert rows["max_examples"].eq("").all() and rows["git_commit"].eq(commit).all()
assert int((rows["direction"].eq("SA") & rows["label_view"].eq("V3-NEU")).sum()) == 6
assert int((rows["direction"].eq("SB") & rows["label_view"].eq("V5-VSMEC")).sum()) == 6
assert int((rows["direction"].eq("SB") & rows["label_view"].eq("V7-VSMEC")).sum()) == 6

sel = json.loads((results_dir / "selection_p1_lr.json").read_text(encoding="utf-8"))
sel_csv = pd.read_csv(results_dir / "p1_lr_selection.csv")
assert sel["protocol_version"] == "phase2-v1" and sel["tie_breaker"] == "smaller_lr"
assert set(sel["selected"].keys()) == {"ViSoBERT/SA/dft", "ViSoBERT/SA/seq", "ViSoBERT/SB/dft", "ViSoBERT/SB/seq"}
assert len(sel_csv) == 4

frozen = json.loads((env_dir / "frozen_before.json").read_text(encoding="utf-8"))
assert all(sha256_file(Path(rel)) == digest for rel, digest in frozen.items()), "Tài nguyên đóng băng bị thay đổi!"

print("Independent verification of all 12 runs, 18 dev rows, test isolation, SKIP & frozen files: PASS")
'''

verify_ctx = json.dumps({
    "output_dir": str(OUTPUT_DIR),
    "results_dir": str(RESULTS_DIR),
    "ckpt_out_dir": str(CKPT_OUT_DIR),
    "env_dir": str(ENV_DIR),
    "commit": COMMIT,
})
run_python(verify_script.replace("__CTX_JSON__", repr(verify_ctx)))


/content/phase2-task07-colab-task06-v1-venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
ViSoBERT_SA_dft_f10_s0_lr3e-05_p2v1_h09960405: SKIP
/content/phase2-task07-colab-task06-v1-venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
ViSoBERT_SA_seq_f10_s0_lr2e-05_p2v1_had5c9c35: SKIP
/content/phase2-task07-colab-task06-v1-venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
ViSoBERT_SB_dft_f10_s0_lr2e-05_p2v1_had5c9c35: SKIP
/content/phase2-task07-colab-task06-v1-venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarni

"/content/phase2-task07-colab-task06-v1-venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).\n  from pandas.core import (\nIndependent verification of all 12 runs, 18 dev rows, test isolation, SKIP & frozen files: PASS\n"

## 10. Sinh báo cáo `task07_lr_sweep.md`, đóng gói ZIP và đồng bộ bàn giao sang Task 08
Tạo báo cáo tổng hợp chi tiết 12 lượt và 4 LR đã khóa, đóng gói toàn bộ kết quả (bắt buộc kèm `work/*/provenance.json` để có thể chạy lại `src.phase2.select_lr` độc lập ở local; mặc định không kèm `resume.pt`), tự kiểm tra danh sách file bên trong gói ZIP và tải về.


In [25]:
report_script = r'''
import json
from pathlib import Path
import pandas as pd

ctx = json.loads(__CTX_JSON__)
output_dir = Path(ctx["output_dir"])
results_dir = Path(ctx["results_dir"])
env_dir = Path(ctx["env_dir"])

env = json.loads((env_dir / "environment.json").read_text(encoding="utf-8"))
sel = json.loads((results_dir / "selection_p1_lr.json").read_text(encoding="utf-8"))
rows = pd.read_csv(results_dir / "phase2_runs.csv", keep_default_na=False)
v7_lookup = {
    r.run_id: float(r.macro_f1)
    for r in rows.loc[rows["label_view"].eq("V7-VSMEC")].itertuples(index=False)
}

run_records = []
for meta_file in sorted((results_dir / "runs").glob("*/metadata.json")):
    rdir = meta_file.parent
    meta = json.loads(meta_file.read_text(encoding="utf-8"))
    metrics = json.loads((rdir / "metrics.json").read_text(encoding="utf-8"))
    cost = json.loads((rdir / "cost.json").read_text(encoding="utf-8"))
    run_records.append({
        "direction": meta["direction"],
        "method": meta["method"],
        "lr": float(meta["lr"]),
        "run_id": meta["run_id"],
        "dev_view": "V3-NEU" if meta["direction"] == "SA" else "V5-VSMEC",
        "dev_score": float(metrics["dev_selection_score"]),
        "v7_dev": v7_lookup.get(meta["run_id"]),
        "best_epoch": metrics["best_epoch"],
        "best_step": metrics["best_step"],
        "steps": cost["actual_update_steps"],
        "early_stopped": bool(metrics["early_stopped"]),
        "train_s": float(cost["train_seconds"]),
        "gpu_h": float(cost["gpu_hours"]),
        "vram_mb": float(cost["peak_vram_mb"]),
    })

order_map = {("SA", "dft"): 0, ("SA", "seq"): 1, ("SB", "dft"): 2, ("SB", "seq"): 3}
run_records.sort(key=lambda r: (order_map[(r["direction"], r["method"])], r["lr"]))

total_train_s = sum(r["train_s"] for r in run_records)
total_gpu_h = sum(r["gpu_h"] for r in run_records)
max_vram_sa = max(r["vram_mb"] for r in run_records if r["direction"] == "SA")
max_vram_sb = max(r["vram_mb"] for r in run_records if r["direction"] == "SB")

mean_h = {}
for d in ("SA", "SB"):
    for m in ("dft", "seq"):
        vals = [r["gpu_h"] for r in run_records if r["direction"] == d and r["method"] == m]
        mean_h[(d, m)] = sum(vals) / len(vals)
est_task08_h = sum(5 * mean_h[(d, "dft")] + 8 * mean_h[(d, "seq")] for d in ("SA", "SB"))

staged = env.get("staged_source_checkpoints", {})
def fmt_ckpt_label(p_str: str) -> str:
    if not p_str:
        return ""
    p = Path(p_str)
    return f"{p.parent.name}/{p.name}" if p.name == "ViSoBERT" else p.name

lines = [
    "# Task07 — Báo cáo dò Learning Rate (12 lượt dev-only, `phase2-v1`)",
    "",
    f"- **Commit:** `{env['commit']}`",
    f"- **Profile môi trường:** `{env['profile_id']}` (session `{env['session_id']}`)",
    f"- **GPU / Precision:** `{env['gpu']}` / `bf16` strict",
    f"- **Python / Torch / CUDA:** `{env['python']}` / `{env['torch']}` / `{env['cuda']}`",
    f"- **Thời gian setup & preflight (trước sweep):** `{env.get('setup_seconds_before_sweep')}s`",
    f"- **Tổng `train_seconds` (12 lượt):** `{total_train_s:.2f}s` (`{total_gpu_h:.4f}` GPU-giờ)",
    f"- **Peak VRAM cực đại:** SA = `{max_vram_sa:.1f} MiB`, SB = `{max_vram_sb:.1f} MiB`",
    f"- **Checkpoint nguồn Stage 1 (`vsmec`, seed 0):** `{fmt_ckpt_label(staged.get('vsmec', {}).get('staged_path', ''))}` (`encoder_sha256={staged.get('vsmec', {}).get('encoder_sha256', '')}`)",
    f"- **Checkpoint nguồn Stage 1 (`neu3`, seed 0):** `{fmt_ckpt_label(staged.get('neu3', {}).get('staged_path', ''))}` (`encoder_sha256={staged.get('neu3', {}).get('encoder_sha256', '')}`)",
    "- **Provenance lịch sử của checkpoint nguồn Stage 1:** `unverified` (theo đúng quy ước `AGENTS.md`).",
    "",
    "## 1. Learning rate được khóa (`selection_p1_lr.json`)",
    "",
    "| Khóa | Chiều | Phương pháp | LR được chọn | Điểm `dev` chọn | Cờ `grid_edge` | `run_id` |",
    "|---|---|---|---:|---:|---|---|",
]
for key in ("ViSoBERT/SA/dft", "ViSoBERT/SA/seq", "ViSoBERT/SB/dft", "ViSoBERT/SB/seq"):
    rec = sel["selected"][key]
    lines.append(
        f"| `{key}` | `{rec['direction']}` | `{rec['method']}` | `{rec['lr']:g}` | "
        f"`{rec['dev_selection_score']:.6f}` | `{rec['grid_edge']}` | `{rec['run_id']}` |"
    )

lines += [
    "",
    "## 2. Kết quả chi tiết cả 12 lượt sweep (`fraction=10`, `seed=0`, `dev`-only)",
    "",
    "| # | Chiều | Phương pháp | LR | View chọn | Điểm chọn `dev` | `V7-VSMEC` `dev` | Best ep / step | Tổng step | Early stop | `train_s` | Peak VRAM (MiB) |",
    "|---:|---|---|---:|---|---:|---:|---|---:|---|---:|---:|",
]
for idx, r in enumerate(run_records, start=1):
    v7_txt = f"{r['v7_dev']:.6f}" if r["v7_dev"] is not None else "—"
    lines.append(
        f"| {idx} | `{r['direction']}` | `{r['method']}` | `{r['lr']:g}` | `{r['dev_view']}` | "
        f"`{r['dev_score']:.6f}` | `{v7_txt}` | `{r['best_epoch']} / {r['best_step']}` | "
        f"`{r['steps']}` | `{r['early_stopped']}` | `{r['train_s']:.2f}` | `{r['vram_mb']:.1f}` |"
    )

lines += [
    "",
    "## 3. Kiểm định & Ghi chú bàn giao",
    "",
    "- **Kiểm định độc lập:** 12 `completion.json`, 12 `provenance.json`, 18 dòng `dev` trong `phase2_runs.csv`, không có `predictions_test.csv`, xác suất hợp lệ, `SKIP`, tài nguyên đóng băng: **PASS**.",
    f"- **Ước lượng sơ bộ thời gian trainer cho Task 08 (26 lượt ở 10% trên cùng loại GPU `{env['gpu']}`):** ~`{est_task08_h:.3f}` GPU-giờ (chưa tính thời gian chấm `test` và lưu checkpoint `main`).",
    "- `train_seconds` không tính thời gian cài đặt môi trường hoặc tải mô hình; nếu có lượt resume thực tế thì `peak_vram_mb` phản ánh lần gọi `fit` cuối.",
]

report_path = output_dir / "reports" / "phase2" / "task07_lr_sweep.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_text = "\n".join(lines) + "\n"
report_path.write_text(report_text, encoding="utf-8")
print(report_text)
'''

run_python(
    report_script.replace(
        "__CTX_JSON__",
        repr(json.dumps({
            "output_dir": str(OUTPUT_DIR),
            "results_dir": str(RESULTS_DIR),
            "env_dir": str(ENV_DIR),
        })),
    )
)

if DRIVE_SESSION_ROOT is not None:
    atomic_copy_file(
        OUTPUT_DIR / "reports" / "phase2" / "task07_lr_sweep.md",
        DRIVE_SESSION_ROOT / "task07_lr_sweep.md",
    )
    print("Đã lưu báo cáo & toàn bộ file kết quả lẻ tại:", DRIVE_SESSION_ROOT)

/content/phase2-task07-colab-task06-v1-venv/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
# Task07 — Báo cáo dò Learning Rate (12 lượt dev-only, `phase2-v1`)

- **Commit:** `edf4067751d72f84ad5e70485614f2d1e71b6d9c`
- **Profile môi trường:** `colab-task06-v1` (session `20261127T113132Z`)
- **GPU / Precision:** `NVIDIA L4` / `bf16` strict
- **Python / Torch / CUDA:** `3.13.18` / `2.14.1+cu130` / `13.0`
- **Thời gian setup & preflight (trước sweep):** `57.077s`
- **Tổng `train_seconds` (12 lượt):** `1045.90s` (`0.2905` GPU-giờ)
- **Peak VRAM cực đại:** SA = `3334.7 MiB`, SB = `2051.4 MiB`
- **Checkpoint nguồn Stage 1 (`vsmec`, seed 0):** `ViSoBERT_Stage1-base-emokeep-bf16_s0_lr5e-5` (`encoder_sha256=3cf5ebc839dca08f73e654a0f5090309c9aa1f515e6e20a9b3e29009a5300805`)
- **Checkpoint nguồn Stage 1 (`neu3`, seed 0):** `ViSoBERT_NEU3-base-emok

### Hướng dẫn bàn giao sau khi tải ZIP về máy local
1. Giải nén gói ZIP bằng `zipfile` của Python (giữ nguyên byte `LF` gốc của artifact/CSV, không chuẩn hóa sang `CRLF`) vào thư mục bằng chứng `results/phase2/task07/<YYYY-MM-DD>_ViSoBERT_sweep12_<GPU>/`.
2. Đặt bản khóa `results/phase2/selection_p1_lr.json` và `results/phase2/p1_lr_selection.csv` (hoặc commit lên nhánh `phase2-resource-transfer` trước Task 08) để truyền vào `--selection-path` cho toàn bộ thực nghiệm chính ở Task 08–10.
3. Lưu lại bản notebook Colab đã chạy có output làm bằng chứng thực thi; không sửa tay các con số trong artifact.
